In [8]:
import pandas as pd

# Load data
df = pd.read_csv(r"C:\Users\vishal singh\Downloads\customer_churn_dataset.csv")

# First 5 rows dekh
print(df.head())

# Data info
print(df.info())

   customer_id  gender  age  tenure contract_type internet_service  \
0            1    Male   34      11       Monthly              NaN   
1            2  Female   26      56        Yearly            Fiber   
2            3    Male   50      67        2-Year              NaN   
3            4    Male   37      29       Monthly              NaN   
4            5    Male   30       9       Monthly            Fiber   

  phone_service  monthly_charges  total_charges payment_method  \
0           Yes            88.03        6273.55  Bank Transfer   
1           Yes            21.59        5655.89           Card   
2           Yes           102.27        6692.22           Card   
3           Yes            83.81         544.09            UPI   
4           Yes            86.67        6005.41           Card   

  paperless_billing  customer_support_calls  number_of_services has_streaming  \
0               Yes                       4                   4            No   
1               Yes 

In [ ]:

# STEP 2: Data Validation


# Missing values check
print("Missing Values:\n", df.isnull().sum())

# Duplicate check
print("\nDuplicate Rows:", df.duplicated().sum())

# Data types check
print("\nData Types:\n", df.dtypes)

Missing Values:
 customer_id                 0
gender                      0
age                         0
tenure                      0
contract_type               0
internet_service          340
phone_service               0
monthly_charges             0
total_charges               0
payment_method              0
paperless_billing           0
customer_support_calls      0
number_of_services          0
has_streaming               0
has_online_security         0
churn                       0
dtype: int64

Duplicate Rows: 0

Data Types:
 customer_id                 int64
gender                        str
age                         int64
tenure                      int64
contract_type                 str
internet_service              str
phone_service                 str
monthly_charges           float64
total_charges             float64
payment_method                str
paperless_billing             str
customer_support_calls      int64
number_of_services          int64
has_streaming  

In [ ]:
# Remove duplicates 
df.drop_duplicates(inplace=True)

# Convert total_charges to numeric 
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')

# Missing values remove 
df.dropna(inplace=True)

print("\n✅ Cleaning Done")


✅ Cleaning Done


In [ ]:

# STEP 3: Feature Engineering

# Avoid division by zero
df = df[df['tenure'] > 0]

# Create new column
df['avg_spend_per_month'] = df['total_charges'] / df['tenure']

# Check output
print(df[['total_charges', 'tenure', 'avg_spend_per_month']].head())

print("\n✅ Feature Engineering Done")

    total_charges  tenure  avg_spend_per_month
1         5655.89      56           100.998036
4         6005.41       9           667.267778
8         2581.88       9           286.875556
9         7667.97      40           191.699250
10         364.05      25            14.562000

✅ Feature Engineering Done


In [ ]:

# STEP 4: Encoding

df_encoded = pd.get_dummies(df, drop_first=True)

print(df_encoded.head())

print("\nTotal Columns After Encoding:", df_encoded.shape[1])
print("\n✅ Encoding Done")

    customer_id  age  tenure  monthly_charges  total_charges  \
1             2   26      56            21.59        5655.89   
4             5   30       9            86.67        6005.41   
8             9   30       9            22.38        2581.88   
9            10   63      40            54.86        7667.97   
10           11   52      25           107.50         364.05   

    customer_support_calls  number_of_services  avg_spend_per_month  \
1                        2                   1           100.998036   
4                        4                   4           667.267778   
8                        5                   5           286.875556   
9                        4                   1           191.699250   
10                       4                   1            14.562000   

    gender_Male  contract_type_Monthly  contract_type_Yearly  \
1         False                  False                  True   
4          True                   True                 False

In [ ]:

# STEP 5: Model Training


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Target column (churn)
X = df_encoded.drop("churn_Yes", axis=1)
y = df_encoded["churn_Yes"]

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model create
model = LogisticRegression(max_iter=1000)

# Train model
model.fit(X_train, y_train)

print("\n✅ Model Training Done")


✅ Model Training Done


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:

# STEP 6: Prediction


# Predict churn (Yes/No)
df['churn_prediction'] = model.predict(X)

# Predict probability (risk score)
df['churn_probability'] = model.predict_proba(X)[:, 1]

# Check output
print(df[['churn', 'churn_prediction', 'churn_probability']].head())

print("\n✅ Prediction Done")

   churn  churn_prediction  churn_probability
1     No             False           0.072653
4    Yes              True           0.823684
8     No             False           0.135487
9     No              True           0.521405
10    No             False           0.179607

✅ Prediction Done


In [ ]:

# STEP 6.5: Automation Setup


def churn_pipeline(input_file, output_file):
    
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    
    # Load data
    df = pd.read_csv(input_file)
    
    # Cleaning
    df.drop_duplicates(inplace=True)
    df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
    df.dropna(inplace=True)
    
    # Feature Engineering
    df = df[df['tenure'] > 0]
    df['avg_spend_per_month'] = df['total_charges'] / df['tenure']
    
    # Encoding
    df_encoded = pd.get_dummies(df, drop_first=True)
    
    # Model
    X = df_encoded.drop("churn_Yes", axis=1)
    y = df_encoded["churn_Yes"]
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)
    
    # Prediction
    df['churn_prediction'] = model.predict(X)
    df['churn_probability'] = model.predict_proba(X)[:, 1]
    
    # Save
    df.to_csv(output_file, index=False)
    
    print("✅ Pipeline executed successfully!")

# Run pipeline
churn_pipeline(r"C:\Users\vishal singh\Downloads\customer_churn_dataset.csv", "final_churn_data.csv")

✅ Pipeline executed successfully!


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:

# FINAL STEP: Export


df.to_csv("final_churn_data.csv", index=False)

print("🎉 Final File Ready for Power BI")

🎉 Final File Ready for Power BI


In [19]:
import os
print(os.listdir())

['.anaconda', '.bash_history', '.conda', '.continuum', '.docker', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.lesshst', '.nbi', '.node_repl_history', '.VirtualBox', '.vishal', '.vscode', 'anaconda_projects', 'AppData', 'Application Data', 'churm.ipynb', 'Contacts', 'Cookies', 'dictnory.py', 'dockervishal', 'Documents', 'Downloads', 'Favorites', 'final_churn_data.csv', 'IntelGraphicsProfiles', 'Links', 'Local Settings', 'Microsoft', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{0e71e67e-f4b4-11ef-b960-a243e52a618f}.TM.blf', 'NTUSER.DAT{0e71e67e-f4b4-11ef-b960-a243e52a618f}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{0e71e67e-f4b4-11ef-b960-a243e52a618f}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'package-lock.json', 'PrintHood', 'PycharmProjects', 'python programing', 'Recent', 'Saved Games', 'Searches', 'SendTo', 'Start Menu', 'Templates', 'TurboC++', 'Untitled Folder', 'U

In [20]:
import os
print(os.path.exists("final_churn_data.csv"))

True
